In [1]:
# ==========================================================
# STUDENT 2
# NOTEBOOK 02
# DATA PREPARATION
# ==========================================================

import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ==========================================================
# PROJECT CONFIGURATION
# ==========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

DATASETS = PROJECT_ROOT / "datasets"
PROCESSED_DATA = DATASETS / "processed"

print("=" * 70)
print("PROJECT CONFIGURATION")
print("=" * 70)
print("Project Root :", PROJECT_ROOT)
print("Processed Data :", PROCESSED_DATA)

PROJECT CONFIGURATION
Project Root : C:\Users\DELL\Documents\Email_Phishing_Project
Processed Data : C:\Users\DELL\Documents\Email_Phishing_Project\datasets\processed


In [3]:
# ==========================================================
# LOAD MASTER DATASET
# ==========================================================

master_file = PROCESSED_DATA / "cleaned_master_email_dataset.csv"

master_df = pd.read_csv(master_file)

print("=" * 70)
print("MASTER DATASET")
print("=" * 70)

print(master_df.shape)

MASTER DATASET
(258656, 6)


In [4]:
# ==========================================================
# VERIFY REQUIRED COLUMNS
# ==========================================================

required = [

    "cleaned_email_text",

    "label"

]

for column in required:

    print(

        f"{column}:",

        column in master_df.columns

    )

cleaned_email_text: True
label: True


In [5]:
# ==========================================================
# DATASET DISTRIBUTION
# ==========================================================

distribution = pd.DataFrame({

    "Count":

    master_df["label"].value_counts()

})

distribution["Percentage"] = (

    distribution["Count"]

    / len(master_df)

    *100

).round(2)

distribution

,Count,Percentage
label,,
0,235464,91.03
1,23192,8.97


In [6]:
train_file = PROCESSED_DATA / "train_dataset.csv"

validation_file = PROCESSED_DATA / "validation_dataset.csv"

test_file = PROCESSED_DATA / "test_dataset.csv"

if (

    train_file.exists()

    and validation_file.exists()

    and test_file.exists()

):

    print("Loading existing split...")

    train_df = pd.read_csv(train_file)

    validation_df = pd.read_csv(validation_file)

    test_df = pd.read_csv(test_file)

else:

    train_df, temp_df = train_test_split(

        master_df,

        test_size=0.30,

        stratify=master_df["label"],

        random_state=SEED

    )

    validation_df, test_df = train_test_split(

        temp_df,

        test_size=0.50,

        stratify=temp_df["label"],

        random_state=SEED

    )

    train_df.to_csv(train_file, index=False)

    validation_df.to_csv(validation_file, index=False)

    test_df.to_csv(test_file, index=False)

print(len(train_df))

print(len(validation_df))

print(len(test_df))

Loading existing split...
181059
38798
38799


In [7]:
# ==========================================================
# VERIFICATION SUBSET
# ==========================================================

TRAIN_SIZE = 5000

VALIDATION_SIZE = 1000

TEST_SIZE = 1000

In [8]:
def stratified_subset(df, sample_size):

    fractions = (
        df["label"]
        .value_counts(normalize=True)
    )

    subsets = []

    for label, fraction in fractions.items():

        n = round(sample_size * fraction)

        subset = (

            df[df["label"] == label]

            .sample(

                n=n,

                random_state=SEED

            )

        )

        subsets.append(subset)

    return (

        pd.concat(subsets)

        .sample(

            frac=1,

            random_state=SEED

        )

        .reset_index(drop=True)

    )

In [9]:
train_subset = stratified_subset(

    train_df,

    TRAIN_SIZE

)

validation_subset = stratified_subset(

    validation_df,

    VALIDATION_SIZE

)

test_subset = stratified_subset(

    test_df,

    TEST_SIZE
)

In [10]:
train_subset.to_csv(

    PROCESSED_DATA /

    "bert_train_subset.csv",

    index=False

)

validation_subset.to_csv(

    PROCESSED_DATA /

    "bert_validation_subset.csv",

    index=False

)

test_subset.to_csv(

    PROCESSED_DATA /

    "bert_test_subset.csv",

    index=False

)

print("=" * 70)
print("SUBSETS CREATED SUCCESSFULLY")
print("=" * 70)

print(train_subset.shape)
print(validation_subset.shape)
print(test_subset.shape)

SUBSETS CREATED SUCCESSFULLY
(5000, 7)
(1000, 7)
(1000, 7)
